# 허위매물 탐지: Recall과 F1을 함께 최적화

허위매물(`허위매물여부=1`)을 양성 클래스로 평가합니다. 개발 데이터의 5-fold OOF 예측에서 **Recall ≥ 0.80 조건을 만족하는 F1 최대값**으로 모델과 임계값을 선택합니다. 최종 20% holdout은 선택을 마친 뒤에만 평가합니다.

이 노트북은 저장된 실험 결과를 바로 읽습니다. 전체 비교를 재실행하려면 아래 `RETRAIN = True`로 변경하세요. 기존 `허위매물 전처리탐지_문제.ipynb`는 초기 분석 기록입니다.

In [ ]:
import json
from pathlib import Path
import joblib
import pandas as pd
from IPython.display import display, Image
from fraud_model import TARGET, ListingFeatures, predict_listings
from train_model import run_training

RETRAIN = False
MIN_RECALL = 0.80
SEED = 20261017
OUTPUT = Path('artifacts')
data = pd.read_csv('train.csv')
display(data[TARGET].value_counts().rename_axis('실제 클래스').to_frame('매물 수'))

## 전처리와 파생 변수

ID를 제외하고, 날짜·결측 여부·면적당 보증금과 월세·층 비율 등을 생성합니다. 이 단계는 각 행만 사용하며 타겟 통계를 만들지 않습니다. 중앙값 대체, 스케일링, 원핫 인코딩은 각 학습 fold에서만 적합합니다. CatBoost는 원래 범주형 문자열과 수치형 결측값을 직접 사용합니다.

새로운 중개사무소나 플랫폼도 동일한 저장된 전처리로 예측합니다.

In [ ]:
preview = ListingFeatures().fit_transform(data.drop(columns=TARGET))
print(f'원본: {data.shape}, 모델 입력 변수: {preview.shape[1]}개')
display(preview.head())

## 모델 선택

LogisticRegression, RandomForest, ExtraTrees, HistGradientBoosting, CatBoost, LightGBM, XGBoost의 25개 설정을 동일한 5개 fold에서 비교합니다. 클래스 가중치, 깊이, 정규화 등을 탐색합니다. 임계값은 실제 OOF 점수 후보 전체에서 선택합니다.

OOF F1 → Recall → Average Precision 순으로 선택하고, 선택된 모델을 개발 데이터 전체로 학습하여 저장합니다. Holdout 평가 결과를 보고 임계값이나 모델을 다시 선택하지 않습니다.

In [ ]:
if RETRAIN or not (OUTPUT / 'metrics.json').exists():
    run_training('train.csv', OUTPUT, min_recall=MIN_RECALL, seed=SEED)
report = json.loads((OUTPUT / 'metrics.json').read_text(encoding='utf-8'))
manifest = json.loads((OUTPUT / 'run_manifest.json').read_text(encoding='utf-8'))
print('저장된 실험 설정:', {key: manifest[key] for key in ['seed', 'folds', 'min_recall']})
comparison = pd.read_csv(OUTPUT / 'model_comparison.csv')
display(comparison.drop_duplicates('family')[['family', 'candidate', 'threshold', 'precision', 'recall', 'f1', 'average_precision', 'default_recall', 'default_f1']].round(4))

## 최종 평가

다음 수치는 모델 선택에 사용하지 않은 holdout에서 계산합니다. `default`와 `tuned`는 **같은 모델**의 임계값 0.5와 OOF 최적 임계값을 비교합니다. `fn`은 놓친 허위매물, `fp`는 허위매물로 잘못 분류한 정상 매물입니다.

OOF 수치는 선택에 사용했으므로 최종 일반화 성능으로 해석하지 않습니다. Recall 80%는 OOF 선택 조건으로, 새 데이터에서 보장되는 하한은 아닙니다. 이 holdout도 기존 프로젝트 데이터의 일부이며 새로운 외부 데이터가 아닙니다.

In [ ]:
print('선택된 모델:', report['selected_candidate'])
print('선택된 임계값:', report['threshold'])
display(pd.DataFrame({'default': report['holdout_default'], 'tuned': report['holdout_tuned']}).T.round(4))
print('Holdout recall 목표 달성:', report['holdout_recall_target_met'])
display(pd.DataFrame(report['holdout_bootstrap_95_ci'], index=['95% 하한', '95% 상한']).round(4))
display(Image(filename=str(OUTPUT / 'evaluation.png')))

## 새 매물 예측

저장된 모델에는 전처리, 분류기, 선택 임계값이 함께 들어 있습니다. `허위매물점수`는 분류 점수이며 실제 사기 확률로 보정된 수치는 아닙니다. 아래는 holdout 행으로 확인하는 예시입니다. 실제 사용 시 `new_listings = pd.read_csv('새매물.csv')`로 바꾸세요. 모델 파일은 신뢰할 수 있는 파일만 불러오세요.

In [ ]:
artifact = joblib.load(OUTPUT / 'fraud_detector.joblib')
new_listings = data.iloc[manifest['holdout_rows'][:10]].drop(columns=TARGET)
display(predict_listings(artifact, new_listings))

## 해석과 다음 검증

- 같은 데이터에서 모델·임계값 후보를 계속 늘리면 OOF 결과에도 과적합할 수 있습니다. 최종 결과가 낮다고 holdout을 보고 다시 튜닝하지 마세요.
- 95% 구간은 고정된 모델과 현재 holdout의 표본 불확실성만 나타내며, 모델 선택 불확실성을 포함하지 않습니다.
- 무작위 분할 성능은 미래 날짜 또는 처음 보는 중개사무소 성능과 다를 수 있습니다. 운영 전 시간순·중개사무소 그룹별 외부 검증이 필요합니다.
- 저장 모델은 평가 때와 같은 개발 데이터 80%로 학습되어 있습니다. 전체 데이터 재학습 모델의 성능으로 혼동하지 마세요.